# 🏠 Airbnb Data Analysis & Visualization (Data Visualization and EDA Project)

## As a popular platform connecting travelers with local accommodations, Airbnb generates a wealth of data. This project undertakes Data Visualization of the Airbnb dataset to extract meaningful insights and understand the dynamics of the housing rental marketplace.

## 🎯 Objective:
To analyze Airbnb listing data using Python to uncover key trends, pricing variations, geographical patterns, and host behaviors.

## ✨ Key Activities:
- Feature-wise visualizations were created to identify important characteristics of the listings.

- These insights are presented through a clear and informative approach.
- The analysis aims to understand the factors influencing pricing, availability, and location of Airbnb rentals.
- The project explores other interesting aspects and relationships within the Airbnb dataset.

***
# 📊 Description of Features in the Dataset:

**NAME** - Represents the name of the listing, providing information about it.

**host id** - This is the unique identification number assigned to each host.

**id** - This is the unique identification number for each listing.

**host\_identity\_verified** - Indicates whether the host's identity has been verified or not.

**host name** - The name of the host who owns the listing.

**neighbourhood group** - Represents the broader geographical area or large neighborhood group.

**neighbourhood** - Represents the specific and smaller neighborhood area within the group.

**lat** - The latitude coordinate of the listing's location.

**long** - The longitude coordinate of the listing's location.

**country** - The country in which the listing is located.

**country code** - The specific code representing the country.

**instant\_bookable** - Indicates whether the listing can be booked instantly by guests.

**cancellation\_policy** - Specifies the terms and conditions regarding cancellations for the listing.

**room type** - Describes the type of accommodation offered (e.g., entire home, private room).

**Construction year** - Specifies the year in which the property or listing was built.

**price** - The rental price of the listing.

**service fees** - Any additional charges, such as cleaning fees, associated with the booking.

**minimum nights** - The minimum number of nights a guest can stay at the listing.

**number of reviews** - The total number of reviews the listing has received on the platform.

**last review** - The date when the listing received its most recent review.

**reviews per month** - The average number of reviews the listing receives per month.

**review rate number** - The rating of the listing, typically on a scale from 0 to 5.

**calculated host listings count** - The total number of listings managed by the host.

**availability 365** - The number of days the listing is available for booking within a 365-day period.

**house\_rules** - The specific rules and guidelines for guests staying at the property.

**license** - Indicates if the host has provided a valid license for the listing.

***

In [ ]:

## importing essential libraries
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
%matplotlib inline
pd.set_option('display.max_columns', None)
# from google.colab import drive
# drive.mount('\content\drive')

raw_df = pd.read_csv('C:\\Users\\HP\\Downloads\\archive\\Airbnb_Open_Data.csv' , low_memory= False)

df = raw_df.copy() ## creating a copy of the original dataset.

In [ ]:
df.head()# checking the top five rows

In [ ]:
df[ df['license'].notnull()] #had some doubt regarding this feature

In [ ]:
print(df.info())
print(df.describe()) ## checking the information

In [ ]:
df.isnull().sum() ## figuring out the sum of null values

In [ ]:
print(df.groupby('NAME').size()) ## here it can be seen that the name feature has a lot of unique categories
print(df['NAME'].nunique()) ## 61281 unique categories are the in this feature

## so, this feature can be dropped

df.drop('NAME', axis = 1, inplace =True)

## doing the same thing for house rules feature
df.drop('house_rules',inplace = True, axis = 1)

In [ ]:
print(df['id'].nunique())
print(df['host id'].nunique()) ## since id and host id are unique I won't drop it

In [ ]:
print('Unique dates: {}'.format(df['last review'].nunique()))
df['last review']= pd.to_datetime(df['last review'])

sorted_dates = sorted(df['last review'].unique(), reverse = True)
print('sorted dates: {}'.format(sorted_dates))## here there are four outliers - 2058-06-16, 2040-06-16, 2025-06-26 and 2026-03-28

print('Max date: {}'.format(df['last review'].max())) ## checking the max value value before filtering

df = df[(df['last review'] != '2058-06-16') & (df['last review'] != '2040-06-16')  & (df['last review'] != '2025-06-26') & (df['last review'] != '2026-03-28') ] ## doing filtering

print('Max date: {}'.format(df['last review'].max()))## max value after filtering

df['monthOfLastReview'] = df['last review'].dt.month ## created new columns which will store the month and day of the last review.
df['dayOfLastReview'] = df['last review'].dt.dayofweek
df['is_weekend'] = df['dayOfLastReview'].apply(lambda x: 1 if x >= 5 else 0) #This column is for the last review itself

In [ ]:
sns.histplot(x = df['last review'].dt.day)
plt.xlabel('lastReviewDay')
plt.show()
sns.histplot(x = df['last review'].dt.month)
plt.show()

## From the above months histogram plot it can be observed that:
- most of the last reviews are done in the month of June which means people come to different locations to spend their holidays and they stay in a rental property. That is the reason why most of the listings has last review on the month of June




In [ ]:
## since price and service fees have $ sign I am removing it
df['service fee'] = pd.to_numeric(df['service fee'].str.replace('[\$,]','', regex = True), errors = 'coerce')
df['price'] = pd.to_numeric(df['price'].str.replace('[\$,]','', regex = True), errors = 'coerce')


In [ ]:
## capturing the numerical features
num_features = [feature for feature in df.columns if df[feature].dtypes != 'O' if feature not in ['id','host id','last review']]
print('Numerical feature: ',num_features)


In [ ]:
## capturing the categorical features
cat_features = [feature for feature in df.columns if df[feature].dtypes == 'O']
print('categorical features: ', cat_features)


In [ ]:
## capturing discrete features
des_features = [feature for feature in num_features if df[feature].nunique() <= 25]
print('discrete features: ',des_features)

In [ ]:
##capturing continuous features
con_features = [feature for feature in num_features if df[feature].nunique() > 25]
print('continuous features: ',con_features)


In [ ]:
## checking the no. of unique categories
for feature in cat_features:
  print("Number of unique categories in: '{}' feature are: {}\n".format(feature, df[feature].nunique()))

In [ ]:
## creating count plot of the categorical features
for feature in cat_features:
  if feature == 'host name' or feature == 'neighbourhood':
    pass
  else:
    sns.countplot(x = df[feature])
    plt.title(feature)
    plt.show()

In [ ]:
highlyAvailNeigh = df['neighbourhood'].value_counts().sort_values(ascending=False)
temp_var =  highlyAvailNeigh.head(10)
print(temp_var)
print('Top 10 higly available neighbourhoods: {}'.format(temp_var.index))


temp_df1 = df[df['neighbourhood'].isin(temp_var.index)]

sns.histplot(x = temp_df1['neighbourhood'])
plt.xticks(rotation = 90)
plt.show()


In [ ]:
top10Neigh= df.groupby('neighbourhood')['price'].median().sort_values(ascending=False)
temp_var1 =  top10Neigh.head(10)
print(temp_var1)
print('Top 10 expensive neighbourhoods are: {}'.format(temp_var1.index))

temp_df2 = df[df['neighbourhood'].isin(temp_var1.index)]

temp_df2.groupby('neighbourhood')['price'].median().plot.bar() ## getting the top 10 expensive neighbourhoods by grouping with median price.
plt.ylabel('Median price')


In [ ]:
df.shape

## From the above plots it can be observed that:

- The no. Verified and unconfirmed host identities is the same, providing renters an option to choose the verified host.

- Country and the country code is US.

- There are instant bookable listings which are equal in count with respect to non instant bookable

- Cancellation policy has strict, moderate and flexible criteria which provide renters to choose any kind of policy specified listing from this variety of policies.

- The no. of shared room listings is very less and hotel room listings are almost rare which be inconvenient for renters to choose airbnb platform. But, people/renters can opt for private or entire room if required.

- Top 10 higly available neighbourhoods are: 'Bedford-Stuyvesant', 'Williamsburg', 'Harlem', 'Bushwick',
       'Hell's Kitchen', 'Upper West Side', 'Upper East Side', 'East Village',
       'Midtown' and 'Crown Heights'

- Top 10 expensive neighbourhoods are: 'New Dorp', 'Chelsea, Staten Island', 'Woodrow', 'Fort Wadsworth',
       'Jamaica Hills', 'Midland Beach', 'Riverdale', 'East Morrisania',
       'Huguenot' and 'Shore Acres. New Dorp is the most expensive neighbourhood.

In [ ]:
##replacing the 'neighbourhood' feature because of misspelled categories and creating a countplot of this feature
df['neighbourhood group'] = df['neighbourhood group'].str.title().str.strip()
df['neighbourhood group'] = df['neighbourhood group'].replace({'Manhatan':'Manhattan','Brookln':'Brooklyn'})


sns.countplot(x = df['neighbourhood group'])
plt.show()

## Observations from the above countplot of neighbourhood group:

- Manhattan and Brooklyn have higher listings available may be due to the high popularity or people may feel good to go to these areas for spending their holidays.



In [ ]:
## creating a new categorical variable excluding host name and neighbourhood and creating a barplot on the basis of price feature's median
newCatFeatures = [feature for feature in df.columns if df[feature].dtypes == 'O' and feature not in ['host name', 'neighbourhood']]
for feature in newCatFeatures:

  df.groupby(feature)['price'].median().plot.bar()
  plt.title(feature)
  plt.ylabel('median of price')
  plt.xticks(rotation = 45)
  plt.show()

## Observations from the above plots:

- Surprisingly, all the categories in a particular feature have almost the median price, suggesting that the change in the category will not affect the price of a listing concerning that particular feature itself.

In [ ]:
for feature in con_features:
  sns.histplot(df[feature],bins = 50)
  plt.title(feature)
  plt.show()


In [ ]:
df['price'].value_counts().sort_index() ## just checking the distribution of the starting price values.

In [ ]:
#pd.set_option('display.max_rows', None)

df['minimum nights'].value_counts().sort_index().reset_index() ## here, it can be observed that the original minimum nights range is between 1 to 365.

## So, filtering the dataset.
df = df[(df['minimum nights'] >= 1) & (df['minimum nights'] <= 365)]
print(df.shape)


In [ ]:

sns.histplot(df['minimum nights'], bins = 40)
plt.show()

In [ ]:
#df.set_option('display.max_rows',None)
df['availability 365'].value_counts().sort_index().reset_index()

## From the above plots following observations can be made:

- From the latitude range of 40.67223592 to 40.7839493, most of the listings are available.

- From the longitude range of -73.99916545 to -73.90012143, most of the listings are available.

- The price distribution ranges between $50 and $1200 has the same count distribution, concluding that variety of listings are available with respect to a customer specific needs.

- The service fees ranges between $10 to $240 has the same count distribution just like price which means if the price increases the service fee will also increase.

- The count of minimum nights stay is quite huge from 1 to 10 minimum nights. This means that most probably the listings which have 1-10 minimum nights stay range are quite popular and mostly preferred by the renters.

- Count of number of reviews is higher from 1 to 45 which means that the majority of listings are recently published. Also, the reviews per month has more count between 1 to 10 which also means that the listings are newly published.

- Most of the hosts have the calculated listings between (1-10).

- Most of the hosts have preferred mentioning the 'availability 365' to 0 because there are 23499 listings available where 'availability _365' is mentioned as 0.

In [ ]:
for feature in con_features:
    sns.boxplot(df[feature])
    plt.show()

In [ ]:
for feature in con_features:
    sns.violinplot(df[feature])
    plt.show()

## From the above plots it can be observed that:

- Outliers present in the features - 'calculated host listings', 'availability 365', 'number of reviews', 'reviews per month' and 'minimum nights' are so much ,that's why I wont filter the dataset on the basis of these outliers because a lot of information can be lost. I will check after checking the model accuracy that whether or not the outliers in these features are having any impact on the model's accuracy.

- Violin plot is showing the same analysis like the histogram plots.

In [ ]:
for feature in des_features:
    sns.histplot(df[feature])
    plt.show()

In [ ]:
for feature in des_features:
    sns.boxplot(df[feature])
    plt.show()

## From the above plots it can be observed that:

- There are listings available that got constructed from 2002 to 2022.

- The count of 'Review rate number' is less concentrated in 1.0 which means unpopular listings are less in number.

- The average ratings of listing is 3.0 in this airbnb data.

- Almost 25% of the last review have given on sundays.

In [ ]:
featuresWithNan = [feature for feature in df.columns if df[feature].isnull().sum() >= 1]
print('Features having NaN values: \n')

for feature in featuresWithNan:
    print(feature)

for feature in featuresWithNan:
    data = df.copy()

    data[feature + ' NaN'] = np.where(data[feature].isnull(), 1,0)
    data.groupby(feature + ' NaN')['price'].median().plot.bar() ## created a copy dataframe to visualize the NaN values and non null values relationship with price median.
    plt.show()


## From the above plots it can be observed that:

- There is no significant change in median price with respect to null and not values. That basically means that the presence of null value is not affecting the average price.

### Doing train test split by considering the price feature as an output feature.

In [ ]:
## removing the features that aren't necessary for model training.
from sklearn.model_selection import train_test_split

X_train, X_test, y_train , y_test = train_test_split(df.drop(['id','host id','host name','last review','price','license'],axis=1) ,df['price'], test_size = 0.2, random_state= 42)

In [ ]:
X_train ## printing the X_train

In [ ]:
y_train## printing y_train

## Now handling the missing values in features of both X and y.

In [ ]:
## First, handling the numerical features:
numericalWithNan = [feature for feature in X_train.columns if X_train[feature].dtypes != 'O' if X_train[feature].isnull().sum() >= 1]
print('Numerical features in X_train: {}'.format(numericalWithNan))

In [ ]:
for feature in numericalWithNan:

    X_median = X_train[feature].median()

    X_train[feature] = X_train[feature].fillna(X_median) ##filling the NaN with X_train's median value for both train and test.
    X_test[feature] = X_test[feature] .fillna(X_median)

print('Missing values sum (of numerical features) in X_train:\n {} \n\n and X_test:\n {}'.format(X_train[numericalWithNan].isnull().sum(), X_test[numericalWithNan].isnull().sum()))

In [ ]:
## Now handling the missing values in categorical features in X:

catWithNan = [feature for feature in X_train.columns if X_train[feature].dtypes == 'O' if X_train[feature].isnull().sum() >= 1]

print('Categorical features having NaN values in X_train are: {}'.format(catWithNan))

for feature in catWithNan:
    X_train[feature] = X_train[feature].fillna('Missing')
    X_test[feature] = X_test[feature].fillna('Missing')

print('\nSum of null values present in each feature of X_train: \n\n{} \n\n and X_test: {}'.format(X_train.isnull().sum(), X_test.isnull().sum()))

## Handling the rare categories in independent features:

In [ ]:
XtrainCatfeatures = [feature for feature in X_train.columns if X_train[feature].dtypes == 'O'] ## making the categorical column for X_train features.

print('Categorical features in X_train: {}'.format(XtrainCatfeatures))

for feature in XtrainCatfeatures:
    temp = X_train[feature].value_counts(normalize = True)
    print(temp)
    common_cat = temp[temp > 0.01].index
    print(common_cat,'\n')

    ## Converting those categories that have less than 1% of the value count into 'Rare_cat'

    X_train[feature] = np.where(X_train[feature].isin(common_cat), X_train[feature], 'Rare_cat')
    X_test[feature] = np.where(X_test[feature].isin(common_cat), X_test[feature], 'Rare_cat')


In [ ]:
for feature in XtrainCatfeatures: ## printing the values counts again to check the percentage of 'Rare_cat' values:
    temp = X_train[feature].value_counts(normalize = True)
    print(temp)
    common_cat = temp[temp > 0.01].index
    print(common_cat,'\n')

#From the above output it can be observed that the 'Rare_cat' in 'neighbourhood' feature has 32% weightage because of there were 225 unique categories present before handling rare categories in this feature.

In [ ]:
temp_df = X_train.join(y_train) ## creating a temporary variable to store the X and Y train
temp_df
temp_df.groupby('neighbourhood')['price'].median().plot.bar()

Since from the above plot it can be observed that the median of 'Rare_cat' is almost the same as compared to other categories in 'neighbourhood' feature, 'Rare_cat' won't affect the price pattern during model training because median is same for all categories with respect to price.

This encoding of the 'Rare_cat' will help the model to overcome overfitting. The model can't learn patterns for all the unique categories especially, the rare ones and because of this reason the model tries to overfit the rare categories. So, by reducing the number of less occurring categories by giving them the label of 'Rare_cat' will make the model faster and efficient.

## Categorical encoding in X_train:

In [ ]:
for feature in XtrainCatfeatures:
    label_order = X_train.join(y_train).groupby(feature)['price'].median().sort_values().index
    label_map = {k : i for i, k in enumerate(label_order, 0)}

    unknown_value = max(label_map.values()) + 1


    X_train[feature] = X_train[feature].map(label_map)
    X_test[feature] = X_test[feature].map(lambda x: label_map.get(x, unknown_value))
map




## Handling missing values in y_train and y_test:

In [ ]:
yTrainMedian = y_train.median()  ## filing the missing value in both y train and test with y_train's median value.
y_train = y_train.fillna(yTrainMedian)
y_test = y_test.fillna(yTrainMedian)


In [ ]:
print('Sum of null values in y_train: {}\n'.format(y_train.isnull().sum())) ## checking the sum of null values in x and y.
print('Sum of null values in y_test: {}\n'.format(y_test.isnull().sum()))

print('Sum of null values in X_train: {}\n'.format(X_train.isnull().sum()))
print('Sum of null values in X_test: {}\n'.format(X_test.isnull().sum()))




## Log transformation:

In [ ]:
skewedFeatures = [feature for feature in X_train.columns if X_train[feature].dtypes != 'O' if (X_train[feature] > 0).all() if X_train[feature].nunique() > 25]
print('Features that are right skewed are: {}'.format(skewedFeatures))



In [ ]:
XtrainLog = X_train.copy() ## Creating a new dataframe where transformed x train and test will be stored.
XtestLog = X_test.copy()

for feature in skewedFeatures:
    XtrainLog[feature] = np.log1p(XtrainLog[feature])
    XtestLog[feature] = np.log1p(XtestLog[feature])

In [ ]:
ytrainLog = np.log1p(y_train)
ytestLog = np.log1p(y_test)


## Feature scaling:

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

XtrainScaled = scaler.fit_transform(XtrainLog)
XtestScaled = scaler.transform(XtestLog)

## Correlation:

In [ ]:
Xtrain_df = pd.DataFrame(XtrainScaled, columns = XtrainLog.columns)

print(Xtrain_df.columns)
corr = Xtrain_df.corrwith(ytrainLog, method = 'spearman')

sns.barplot(x = corr.index, y = corr.values)
plt.title('Corrrelation of features with price')
plt.xticks(rotation = 90)
plt.show()

In [ ]:
def tag_corr(val):
    if abs(val) >= 0.05:
        return 'Strong'
    elif abs(val) >= 0.02:
        return 'Moderate'
    elif abs(val) >= 0.005:
        return 'Weak'
    else:
        return 'No Corr'
    
corr_df = pd.DataFrame({'Feature': corr.index, 'Correlation' : corr.values})
corr_df['Tag'] = corr_df['Correlation'].apply(tag_corr)
corr_df

## From the above correlation it can be observed that the correlation of features with price is negligible.

### Feature Selection:

In [ ]:
from sklearn.linear_model import Lasso
from sklearn.feature_selection import SelectFromModel

feature_sel_model = SelectFromModel(Lasso(alpha = 0.005, random_state = 0))
feature_sel_model = feature_sel_model.fit(XtrainScaled, y_train)
selectedFeatures = XtrainLog.columns[feature_sel_model.get_support()]

print('Total number of features: {}\n'.format(XtrainLog.shape[1]))
print('Number of selected features: {}\n'.format(len(selectedFeatures)))
print('Selected features: {}\n'.format(selectedFeatures))     ## These features can now be used for model training
print('Features with coefficient shrank to 0: {}'.format(np.sum(feature_sel_model.estimator_.coef_ ==0)))

